# 🪔 Diwali Sales Analysis

**Objective:** Analyze customer purchase behavior during the Diwali festival season to identify key customer segments, geographic patterns, and top-selling product categories.

---

## 1. 📥 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Display settings
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100

os.makedirs('assets', exist_ok=True)
print("Libraries imported successfully.")

## 2. 📂 Load Dataset

In [ ]:
df = pd.read_csv('data/Diwali_Sales_Data.csv', encoding='unicode_escape')
print(f"Dataset shape: {df.shape}")
df.head(10)

In [ ]:
df.info()

## 3. 🧹 Data Cleaning

### 3.1 Drop Irrelevant Columns
`Status` and `unnamed1` are entirely null — we drop them.

In [ ]:
df.drop(columns=['Status', 'unnamed1'], inplace=True)
print(f"Remaining columns: {list(df.columns)}")

### 3.2 Handle Missing Values

In [ ]:
print("Null counts:\n", df.isnull().sum())
df.dropna(inplace=True)
print(f"\nShape after dropping nulls: {df.shape}")

### 3.3 Fix Data Types

In [ ]:
df['Amount'] = df['Amount'].astype(int)
print("Amount dtype:", df['Amount'].dtype)

### 3.4 Summary Statistics

In [ ]:
df[['Age', 'Orders', 'Amount']].describe()

---
## 4. 📊 Exploratory Data Analysis

### 4.1 Gender Analysis

In [ ]:
PINK, BLUE = '#FF8FAB', '#6BAED6'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Gender-wise Purchase Analysis', fontsize=16, fontweight='bold', y=1.02)

# Count by gender
ax = sns.countplot(x='Gender', data=df, palette=[PINK, BLUE], ax=axes[0], order=['F','M'])
for bars in ax.containers: ax.bar_label(bars, fontsize=11, fontweight='bold')
axes[0].set_title('Number of Buyers by Gender')
axes[0].set_xticklabels(['Female', 'Male'])

# Revenue by gender
sales_gen = df.groupby('Gender', as_index=False)['Amount'].sum().sort_values('Amount', ascending=False)
sales_gen['Gender_Label'] = sales_gen['Gender'].map({'F':'Female','M':'Male'})
bars2 = axes[1].bar(sales_gen['Gender_Label'], sales_gen['Amount'], color=[PINK, BLUE], width=0.4, edgecolor='white')
for bar in bars2:
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+200000,
                 f'\u20b9{bar.get_height()/1e6:.1f}M', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('Total Sales Amount by Gender')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\u20b9{x/1e6:.0f}M'))

plt.tight_layout()
plt.savefig('assets/01_gender_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\u2705 Insight: Female customers outnumber male buyers and contribute more to total sales.")

### 4.2 Age Group Analysis

In [ ]:
age_order = ['0-17','18-25','26-35','36-45','46-50','51-55','55+']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Age Group-wise Purchase Analysis', fontsize=16, fontweight='bold', y=1.02)

ax = sns.countplot(data=df, x='Age Group', hue='Gender', order=age_order,
                   palette=[PINK, BLUE], ax=axes[0])
for bars in ax.containers: ax.bar_label(bars, fontsize=9, fontweight='bold')
axes[0].set_title('Buyer Count by Age Group & Gender')
axes[0].legend(labels=['Female','Male'])

colors_age = sns.color_palette('RdYlGn', len(age_order))
sales_age = df.groupby('Age Group', as_index=False)['Amount'].sum()
sales_age = sales_age.set_index('Age Group').reindex(age_order).reset_index()
bars3 = axes[1].bar(sales_age['Age Group'], sales_age['Amount'], color=colors_age, edgecolor='white')
for bar in bars3:
    h = bar.get_height()
    axes[1].text(bar.get_x()+bar.get_width()/2, h+150000,
                 f'\u20b9{h/1e6:.1f}M', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Total Revenue by Age Group')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\u20b9{x/1e6:.0f}M'))

plt.tight_layout()
plt.savefig('assets/02_age_group_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\u2705 Insight: Buyers aged 26-35 (especially females) are the highest spenders.")

### 4.3 State-wise Analysis

In [ ]:
PALETTE_10 = ['#E63946','#457B9D','#2A9D8F','#E9C46A','#F4A261',
               '#264653','#90C8AC','#6D597A','#FFB703','#023047']

fig, axes = plt.subplots(2, 1, figsize=(16, 12))
fig.suptitle('State-wise Purchase Analysis', fontsize=16, fontweight='bold')

top_orders = df.groupby('State', as_index=False)['Orders'].sum().sort_values('Orders', ascending=False).head(10)
sns.barplot(data=top_orders, x='State', y='Orders', hue='State', palette=PALETTE_10, legend=False, ax=axes[0])
for i, (_, row) in enumerate(top_orders.iterrows()):
    axes[0].text(i, row['Orders']+20, str(int(row['Orders'])), ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Top 10 States by Number of Orders')
axes[0].tick_params(axis='x', rotation=20)

top_amount = df.groupby('State', as_index=False)['Amount'].sum().sort_values('Amount', ascending=False).head(10)
sns.barplot(data=top_amount, x='State', y='Amount', hue='State', palette=PALETTE_10, legend=False, ax=axes[1])
for i, (_, row) in enumerate(top_amount.iterrows()):
    axes[1].text(i, row['Amount']+200000, f'\u20b9{row["Amount"]/1e6:.1f}M', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Top 10 States by Total Revenue')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\u20b9{x/1e6:.0f}M'))
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('assets/03_state_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\u2705 Insight: UP, Maharashtra, and Karnataka lead in both orders and revenue.")

### 4.4 Marital Status Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Marital Status-wise Purchase Analysis', fontsize=16, fontweight='bold', y=1.02)

ms_counts = df['Marital_Status'].value_counts().rename(index={0:'Single', 1:'Married'})
bars4 = axes[0].bar(ms_counts.index, ms_counts.values, color=['#6BCB77','#F4A261'], width=0.4, edgecolor='white')
for bar in bars4:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+80,
                 str(int(bar.get_height())), ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Buyer Count by Marital Status')

sales_ms = df.groupby(['Marital_Status','Gender'], as_index=False)['Amount'].sum()
sales_ms['MS_Label'] = sales_ms['Marital_Status'].map({0:'Single', 1:'Married'})
sales_ms['Gender_Label'] = sales_ms['Gender'].map({'F':'Female','M':'Male'})
sns.barplot(data=sales_ms, x='MS_Label', y='Amount', hue='Gender_Label',
            palette=[PINK, BLUE], ax=axes[1])
axes[1].set_title('Revenue by Marital Status & Gender')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\u20b9{x/1e6:.0f}M'))
axes[1].legend(title='Gender')

plt.tight_layout()
plt.savefig('assets/04_marital_status_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\u2705 Insight: Married women have the highest purchasing power.")

### 4.5 Occupation Analysis

In [ ]:
PALETTE_15 = ['#E63946','#457B9D','#2A9D8F','#E9C46A','#F4A261','#264653',
               '#90C8AC','#6D597A','#FFB703','#023047','#BB9999','#6BCB77',
               '#EAAC8B','#FFD6A5','#00ADB5']

fig, axes = plt.subplots(2, 1, figsize=(20, 12))
fig.suptitle('Occupation-wise Purchase Analysis', fontsize=16, fontweight='bold')

occ_count = df.groupby('Occupation').size().reset_index(name='Count').sort_values('Count', ascending=False)
sns.barplot(data=occ_count, x='Occupation', y='Count', hue='Occupation',
            palette=PALETTE_15[:len(occ_count)], legend=False, ax=axes[0])
for i, (_, row) in enumerate(occ_count.iterrows()):
    axes[0].text(i, row['Count']+20, str(int(row['Count'])), ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Number of Buyers by Occupation')
axes[0].tick_params(axis='x', rotation=40)

occ_amt = df.groupby('Occupation', as_index=False)['Amount'].sum().sort_values('Amount', ascending=False)
sns.barplot(data=occ_amt, x='Occupation', y='Amount', hue='Occupation',
            palette=PALETTE_15[:len(occ_amt)], legend=False, ax=axes[1])
for i, (_, row) in enumerate(occ_amt.iterrows()):
    axes[1].text(i, row['Amount']+150000, f'\u20b9{row["Amount"]/1e6:.1f}M', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Total Revenue by Occupation')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\u20b9{x/1e6:.0f}M'))
axes[1].tick_params(axis='x', rotation=40)

plt.tight_layout()
plt.savefig('assets/05_occupation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\u2705 Insight: IT, Healthcare, and Aviation sector customers are the highest spenders.")

### 4.6 Product Category Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(20, 12))
fig.suptitle('Product Category Analysis', fontsize=16, fontweight='bold')

cat_count = df.groupby('Product_Category').size().reset_index(name='Count').sort_values('Count', ascending=False)
sns.barplot(data=cat_count, x='Product_Category', y='Count', hue='Product_Category',
            palette=PALETTE_15[:len(cat_count)], legend=False, ax=axes[0])
for i, (_, row) in enumerate(cat_count.iterrows()):
    axes[0].text(i, row['Count']+15, str(int(row['Count'])), ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('Number of Orders by Product Category')
axes[0].tick_params(axis='x', rotation=40)

cat_amt = df.groupby('Product_Category', as_index=False)['Amount'].sum().sort_values('Amount', ascending=False).head(10)
sns.barplot(data=cat_amt, x='Product_Category', y='Amount', hue='Product_Category',
            palette=PALETTE_10, legend=False, ax=axes[1])
for i, (_, row) in enumerate(cat_amt.iterrows()):
    axes[1].text(i, row['Amount']+200000, f'\u20b9{row["Amount"]/1e6:.1f}M', ha='center', fontsize=9, fontweight='bold')
axes[1].set_title('Top 10 Product Categories by Revenue')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\u20b9{x/1e6:.0f}M'))
axes[1].tick_params(axis='x', rotation=40)

plt.tight_layout()
plt.savefig('assets/06_product_category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\u2705 Insight: Food, Clothing, and Electronics are the top-selling categories.")

### 4.7 Top 10 Best-Selling Products

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
top_prods = df.groupby('Product_ID', as_index=False)['Orders'].sum().sort_values('Orders', ascending=False).head(10)
sns.barplot(data=top_prods, x='Product_ID', y='Orders', hue='Product_ID',
            palette=PALETTE_10, legend=False, ax=ax)
for i, (_, row) in enumerate(top_prods.iterrows()):
    ax.text(i, row['Orders']+5, str(int(row['Orders'])), ha='center', fontsize=10, fontweight='bold')
ax.set_title('Top 10 Best-Selling Products by Orders', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('assets/07_top_products.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.8 Zone Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Zone-wise Distribution', fontsize=16, fontweight='bold', y=1.02)

zone_count = df['Zone'].value_counts()
axes[0].pie(zone_count, labels=zone_count.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(zone_count)), startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[0].set_title('Buyers by Zone')

zone_amt = df.groupby('Zone')['Amount'].sum()
axes[1].pie(zone_amt, labels=zone_amt.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(zone_amt)), startangle=140,
            wedgeprops=dict(edgecolor='white', linewidth=1.5))
axes[1].set_title('Revenue by Zone')

plt.tight_layout()
plt.savefig('assets/08_zone_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. 🏁 Conclusion

> **Married women aged 26–35 from Uttar Pradesh, Maharashtra, and Karnataka, working in IT, Healthcare, and Aviation, are the most likely customers to purchase Food, Clothing, and Electronics during the Diwali festival season.**

### Business Recommendations

| Area | Recommendation |
|------|----------------|
| **Targeting** | Focus ad campaigns on married women aged 26–35 |
| **Geography** | Prioritize inventory & delivery in UP, Maharashtra, Karnataka |
| **Channels** | Leverage professional networks to reach IT & Healthcare workers |
| **Categories** | Ensure strong stock and offers on Food, Clothing, and Electronics |
| **Timing** | Run promotions 2–3 weeks before Diwali to capture peak demand |